# The 3D U-Net: recovering the web in three dimensions

Slices were a tractable warm-up, but the cosmic web is genuinely 3D — a filament threads through a volume, and a 2D cut can't see where it goes. This notebook promotes the network to 3D: every `Conv2d`/`MaxPool2d`/`ConvTranspose2d` becomes its 3D counterpart, and instead of slices we train on **sub-cubes** (patches) of the field. Otherwise the recipe is identical — sparse Poisson tracers in, four-class web out.

This is the step where a GPU matters. The notebook auto-detects CUDA and uses it; on CPU it still runs (the small default config trains in under a minute on a multi-core machine), just slower. If you have an NVIDIA GPU under WSL2, install the CUDA build of PyTorch (plain `pip install torch`, without the CPU index line from `requirements.txt`) and this will use it automatically.

## 0. Setup

In [ ]:
# torch is already in the env; this is a no-op unless missing.
!pip install torch

In [ ]:
import os, time, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(1); np.random.seed(1); torch.set_num_threads(os.cpu_count() or 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- config ---
N         = 64       # field grid per side
L         = 250.0    # box, Mpc/h
patch     = 32       # sub-cube size (must be divisible by 4); the U-Net's input
stride    = 16       # patch spacing (16 = 50% overlap -> more training cubes)
sigma_lin = 2.0      # 2LPT strength
nbar      = 2.0      # mean tracers per voxel
f         = 12       # U-Net width
epochs    = 12
CLASS_NAMES  = ['void','sheet','filament','knot']
CLASS_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
print("torch", torch.__version__, "| device:", device, "| threads", torch.get_num_threads())

## 1. Field, tracers, labels, and 3D patches

The field generator is the 2LPT one from before. `patches()` chops the cube into overlapping sub-cubes — each is one training sample, so the network learns local 3D structure.

In [ ]:
def Pk_bbks(k, ns=0.96, G=0.2):
    k=np.asarray(k,float); q=np.where(k>0,k/G,1e-10)
    T=(np.log(1+2.34*q)/(2.34*q))*(1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**(-0.25)
    return k**ns*T**2
def cic(pos,N,L):
    g=np.zeros((N,N,N)); s=(pos%L)/(L/N); i=np.floor(s).astype(int)%N; fr=s-np.floor(s)
    for dx in (0,1):
        for dy in (0,1):
            for dz in (0,1):
                wx=fr[:,0] if dx else 1-fr[:,0]; wy=fr[:,1] if dy else 1-fr[:,1]; wz=fr[:,2] if dz else 1-fr[:,2]
                np.add.at(g,((i[:,0]+dx)%N,(i[:,1]+dy)%N,(i[:,2]+dz)%N),wx*wy*wz)
    return g
def lpt_delta(N,L,sigma_lin,seed,order=2):
    kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0
    rng=np.random.default_rng(seed)
    dk=fftn(rng.standard_normal((N,N,N)))*np.sqrt(Pk_bbks(np.sqrt(k2))); dk[0,0,0]=0
    d=ifftn(dk).real; d*=sigma_lin/d.std(); dk=fftn(d); src=dk.copy()
    if order==2:
        pij=lambda a,b: ifftn(Kv[a]*Kv[b]/k2*dk).real
        pxx,pyy,pzz=pij(0,0),pij(1,1),pij(2,2); pxy,pxz,pyz=pij(0,1),pij(0,2),pij(1,2)
        S2=pxx*pyy+pxx*pzz+pyy*pzz-pxy**2-pxz**2-pyz**2
        src=dk+(3.0/7.0)*fftn(S2)
    disp=np.array([ifftn(1j*Kv[i]*src/k2).real for i in range(3)])
    q=np.arange(N)*(L/N); QX,QY,QZ=np.meshgrid(q,q,q,indexing='ij'); Q=[QX,QY,QZ]
    pos=np.stack([(Q[i]+disp[i]).ravel() for i in range(3)],axis=1)
    rho=cic(pos,N,L); return rho/rho.mean()-1.0
def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real
def tweb(d,L,lam=0.3):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int64)
def make_cube(N,L,seed,nbar,sigma_lin,order=2):
    d=lpt_delta(N,L,sigma_lin,seed,order); lab=tweb(smooth(d,L,3.0),L,0.3)
    rng=np.random.default_rng(seed+999)
    counts=rng.poisson(nbar*np.clip(1+d,0,None)); dsp=counts/counts.mean()-1.0
    return dsp.astype(np.float32),lab
def patches(vol,lab,p,stride):
    N=vol.shape[0]; X,Y=[],[]
    for i in range(0,N-p+1,stride):
        for j in range(0,N-p+1,stride):
            for k in range(0,N-p+1,stride):
                X.append(vol[i:i+p,j:j+p,k:k+p]); Y.append(lab[i:i+p,j:j+p,k:k+p])
    return np.array(X),np.array(Y)
print("ready")

## 2. The 3D U-Net

Structurally identical to the 2D version — two down blocks, a bottleneck, two up blocks with skips — but with 3D convolutions, so it learns features that span all three axes.

In [ ]:
def double_conv(ci,co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True),
                         nn.Conv3d(co,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True))
class UNet3D(nn.Module):
    def __init__(self,f=12,nc=4):
        super().__init__(); self.e1=double_conv(1,f); self.e2=double_conv(f,2*f); self.b=double_conv(2*f,4*f)
        self.u2=nn.ConvTranspose3d(4*f,2*f,2,2); self.d2=double_conv(4*f,2*f)
        self.u1=nn.ConvTranspose3d(2*f,f,2,2); self.d1=double_conv(2*f,f); self.o=nn.Conv3d(f,nc,1); self.p=nn.MaxPool3d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)
def per_class_f1(pred,true,nc=4):
    o=[]
    for c in range(nc):
        tp=((pred==c)&(true==c)).sum(); fp=((pred==c)&(true!=c)).sum(); fn=((pred!=c)&(true==c)).sum()
        o.append(2*tp/(2*tp+fp+fn+1e-9))
    return o
print(UNet3D(f).__class__.__name__, sum(p.numel() for p in UNet3D(f).parameters()), "parameters")

## 3. Train

Two independent fields (different seeds) for train and validation, class-weighted cross-entropy, Adam. Tensors and model live on `device`.

In [ ]:
dtr,ltr = make_cube(N,L,1,nbar,sigma_lin); dva,lva = make_cube(N,L,2,nbar,sigma_lin)
Xtr,Ytr = patches(dtr,ltr,patch,stride);     Xva,Yva = patches(dva,lva,patch,stride)
mu,sd = Xtr.mean(), Xtr.std()
Xt=torch.tensor(((Xtr-mu)/sd)[:,None]).to(device); Yt=torch.tensor(Ytr).to(device)
Xv=torch.tensor(((Xva-mu)/sd)[:,None]).to(device); Yv=torch.tensor(Yva).to(device)
fr=np.bincount(Ytr.ravel(),minlength=4); w=(1.0/(fr+1)); w=(w/w.sum()*4).astype(np.float32)
print(f"train cubes {Xtr.shape[0]}  shape {tuple(Xtr.shape[1:])}  classmix% {(100*fr/fr.sum()).round(1)}")

net=UNet3D(f).to(device); opt=torch.optim.Adam(net.parameters(),1e-3)
crit=nn.CrossEntropyLoss(weight=torch.tensor(w).to(device))
bs,n=4,len(Xt); t0=time.time()
for ep in range(epochs):
    net.train(); perm=torch.randperm(n)
    for i in range(0,n,bs):
        idx=perm[i:i+bs]; opt.zero_grad(); loss=crit(net(Xt[idx]),Yt[idx]); loss.backward(); opt.step()
    if ep%3==2 or ep==epochs-1:
        net.eval()
        with torch.no_grad(): pv=net(Xv).argmax(1)
        acc=(pv==Yv).float().mean().item(); F=per_class_f1(pv.cpu().numpy(),Yv.cpu().numpy())
        print(f"epoch {ep+1:2d}  loss {loss.item():.3f}  valAcc {acc:.3f}   "
              f"F1 void {F[0]:.2f} sheet {F[1]:.2f} filament {F[2]:.2f} knot {F[3]:.2f}   ({time.time()-t0:.0f}s)")
maj=fr.argmax(); print(f"\nbaseline (all '{CLASS_NAMES[maj]}') valAcc = {(Yva==maj).mean():.3f}")

## 4. See it work

A central slice through one validation sub-cube: the sparse tracers in, the true 3D web, and the network's 3D reconstruction. (The prediction is a full 3D volume — this is just one cut through it.)

In [ ]:
net.eval()
with torch.no_grad(): pv=net(Xv).argmax(1).cpu().numpy()
cmap=ListedColormap(CLASS_COLORS); norm=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)
pidx, zc = 0, patch//2
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(Xva[pidx,zc].T,origin='lower',cmap='inferno'); ax[0].set_title('sparse tracers (slice of 3D cube)')
ax[1].imshow(Yva[pidx,zc].T,origin='lower',cmap=cmap,norm=norm); ax[1].set_title('true web (3D)')
im=ax[2].imshow(pv[pidx,zc].T,origin='lower',cmap=cmap,norm=norm); ax[2].set_title('3D U-Net prediction')
cb=plt.colorbar(im,ax=ax[2],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## Notes and where to go

- **Data is now the limiter.** With one cube split into a couple dozen sub-cubes, the rare knot class is starved — its F1 trails the others. The fixes all scale data: raise `N`, shrink `stride` for more overlap, or generate several realizations (different seeds) and pool their patches. This is exactly what a GPU buys you — bigger `N`, more cubes, a wider net.
- **The honest endpoint.** Train on *real* QUIJOTE fields: load a `df_m_*.npy` via Route A (or read a snapshot with h5py + hdf5plugin), label it with the same T-web, cut patches, and either fine-tune this net or test how a sim-trained net transfers to it. That closes the loop from toy field to real cosmology.
- **Validate against the official void finder.** Once Pylians is in the WSL env, compare these U-Net voids to its spherical void catalog — a real check on what the network is finding.